# BRFSS 2024 — Full Pipeline (editable, all code inline)

This notebook is the **complete pipeline written out in cells** — cleaning, recoding,
EDA, modelling, and evaluation. Unlike `BRFSS_Colab.ipynb` (which just runs the `.py`
files with `!python`), **here you can read and edit every step directly**. Change a
threshold, a model setting, or a chart, then re-run that cell.

**How to use it**
1. Run the setup cell once (clones the repo to get the raw data + installs packages).
2. Then run the stages top to bottom: **Clean → Recode → EDA → Model → Evaluate**.
   Each stage reads the file the previous stage saved, so run them in order the first time.
3. The **⚙️ knobs** cells hold the choices you'd most likely want to change
   (which codes count as "yes", model settings, income cut-off, etc.). Edit those and re-run.

> Target: `mental_risk = 1` when a respondent reports **≥ 14** poor-mental-health days
> in the past 30 (`MENTHLTH`).

> **Note on syncing:** this is a standalone editable copy. The authoritative pipeline is
> still the `.py` scripts in the repo. If you make a change here you want to keep, tell
> the notebook's owner so it gets copied back into the scripts.


## Step 0 — Setup: get the data & install packages

Clones the GitHub repo (the raw survey file and committed data come with it) and installs
the two libraries Colab may be missing. Safe to re-run.

In [ ]:
import os

REPO_URL = 'https://github.com/Kudos-4/BRFSS.git'
REPO_DIR = '/content/BRFSS'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

%cd {REPO_DIR}

# Colab already has pandas / numpy / scikit-learn / matplotlib / seaborn.
!pip install -q xgboost pyarrow
print('\nSetup done. Working directory:', os.getcwd())

---
## Stage 1 — Clean the raw survey  (`BRFSS.py`)

Loads only the ~16 columns we use, then:
- recodes **88 → 0** for the health-day counts (88 means *"none / zero bad days"*, i.e. valid),
- drops rows that are blank (`NaN`) or carry a *"no valid response"* code from the codebook,
- builds the **`mental_risk`** target.

Saves `BRFSS_Cleaned/BRFSS_2024_cleaned.parquet`.

**⚙️ Knobs — which codes mean "no valid response".**
Each number listed causes a row to be dropped (the respondent gave no usable answer).
`88` is deliberately *not* here — it's recoded to 0 above because it means zero bad days.

In [ ]:
import pandas as pd
import os

# Paths are relative to the repo root (works in Colab after Step 0).
PARQUET_PATH = "BRFSS_2024.parquet"
OUTPUT_DIR   = "BRFSS_Cleaned"

# Columns of interest + their "no valid response" codes (drop those rows).
INVALID_CODES = {
    'MENTHLTH': [77, 99],    # Days mental health not good  (88=None->0)
    'PHYSHLTH': [77, 99],    # Days physical health not good (88=None->0)
    'EXERANY2': [7, 9],      # Did any physical activity
    '_TOTINDA': [9],         # Meets activity guidelines
    '_SMOKER3': [9],         # Smoking status
    '_DRNKWK3': [99900],     # Drinks per week
    '_BMI5':    [],          # BMI x100 -- blank only, no numeric sentinel
    '_BMI5CAT': [],          # BMI category -- blank only
    '_AGE80':   [],          # Imputed age -- no refusal codes
    'SEXVAR':   [],          # Sex -- no refusal codes recorded
    'EDUCA':    [9],         # Education level
    'INCOME3':  [77, 99],    # Income (was INCOME2 pre-2024)
    'EMPLOY1':  [9],         # Employment status
    '_HLTHPL2': [9],         # Health insurance (was HLTHPLN1 pre-2024)
    'PERSDOC3': [7, 9],      # Personal doctor (was PERSDOC2 pre-2024)
    'CHECKUP1': [7, 9],      # Last checkup  (8=Never is a valid answer)
}
COLUMNS = list(INVALID_CODES.keys())

In [ ]:
print("=" * 70)
print("BRFSS 2024 -- DATA CLEANING")
print("=" * 70)

# 1. Load only the columns we need
df = pd.read_parquet(PARQUET_PATH, columns=COLUMNS)
n_original = len(df)
print(f"Loaded : {n_original:,} rows x {df.shape[1]} columns")

# 2. Recode 88 ('None / zero days') -> 0 for the day-count variables
for col in ['MENTHLTH', 'PHYSHLTH']:
    n = (df[col] == 88).sum()
    df[col] = df[col].replace(88, 0)
    print(f"  {col}: {n:,} rows recoded 88 -> 0")

# 3. Drop rows blank / not asked (NaN)
before = len(df)
df = df.dropna(subset=COLUMNS)
print(f"  Removed {before - len(df):,} NaN rows -> {len(df):,} remain")

# 4. Drop rows with "no valid response" codes
for col, bad in INVALID_CODES.items():
    if not bad:
        continue
    mask = df[col].isin(bad)
    if mask.sum():
        df = df[~mask]
        print(f"  {col:<12} codes {str(bad):<12} removed {mask.sum():,} rows")

n_final = len(df)

# 5. Create the binary target
df['mental_risk'] = (df['MENTHLTH'] >= 14).astype(int)

# 6. Summary
c0 = (df['mental_risk'] == 0).sum()
c1 = (df['mental_risk'] == 1).sum()
print("\n" + "=" * 70)
print(f"  Usable rows : {n_final:,}  ({n_final/n_original*100:.1f}% of original)")
print(f"  mental_risk=0 (< 14 days) : {c0:,}  ({c0/n_final*100:.1f}%)")
print(f"  mental_risk=1 (>=14 days) : {c1:,}  ({c1/n_final*100:.1f}%)")

# 7. Save
os.makedirs(OUTPUT_DIR, exist_ok=True)
df.to_parquet(f"{OUTPUT_DIR}/BRFSS_2024_cleaned.parquet", index=False)
df.to_csv(f"{OUTPUT_DIR}/BRFSS_2024_cleaned.csv", index=False)
print(f"\nSaved {OUTPUT_DIR}/BRFSS_2024_cleaned.parquet")
df.head()

---
## Stage 2 — Recode into binary features  (`BRFSS_Recode.py`)

The PI's five prep steps: (1) count unique values, (2) list them, (3) drop any leftover
`NaN`, (4–5) turn multi-category survey codes into clean **0/1** features.

Saves `BRFSS_Cleaned/BRFSS_2024_recoded.parquet`.

**⚙️ Knobs — the binary recodes.** This is where the judgment calls live. Each row maps a
**new 0/1 column ← (source column, the codes that become 1)**. Everything else becomes 0.
Edit a code set to change a definition — e.g. make `low_income` `<$35k` by adding `5`, or
count "ever smokers" by adding `3` to `current_smoker`.

In [ ]:
# Human-readable label for each raw column (from the codebook)
VAR_LABELS = {
    'MENTHLTH': 'Poor mental health days (0-30)',
    'PHYSHLTH': 'Poor physical health days (0-30)',
    'EXERANY2': 'Any physical activity  (1=Yes, 2=No)',
    '_TOTINDA': 'Meets activity guidelines  (1=Yes, 2=No)',
    '_SMOKER3': 'Smoker status  (1=everyday, 2=someday, 3=former, 4=never)',
    '_DRNKWK3': 'Alcoholic drinks per week',
    '_BMI5':    'BMI x100',
    '_BMI5CAT': 'BMI category  (1=under, 2=normal, 3=over, 4=obese)',
    '_AGE80':   'Age (18-80)',
    'SEXVAR':   'Sex  (1=Male, 2=Female)',
    'EDUCA':    'Education  (1..6, 6=college grad)',
    'INCOME3':  'Income bracket  (1=<$10k .. 11=$200k+)',
    'EMPLOY1':  'Employment  (1=wages,2=self,3-4=out of work,5=homemaker,6=student,7=retired,8=unable)',
    '_HLTHPL2': 'Health insurance  (1=Have, 2=None)',
    'PERSDOC3': 'Personal doctor  (1=one, 2=>one, 3=No)',
    'CHECKUP1': 'Last checkup  (1=<1yr, 2=<2yr, 3=<5yr, 4=5yr+, 8=never)',
    'mental_risk': 'TARGET: mental health risk (MENTHLTH>=14)',
}

# new column          source       codes -> 1      meaning of "1"
BINARY_RECODES = {
    'exercise':          ('EXERANY2', {1},          'did any physical activity'),
    'meets_activity':    ('_TOTINDA', {1},          'meets activity guidelines'),
    'current_smoker':    ('_SMOKER3', {1, 2},       'smokes every/some days'),
    'obese':             ('_BMI5CAT', {4},          'BMI category = obese'),
    'female':            ('SEXVAR',   {2},          'female'),
    'insured':           ('_HLTHPL2', {1},          'has health insurance'),
    'has_pers_doctor':   ('PERSDOC3', {1, 2},       'has >=1 personal doctor'),
    'checkup_within_1yr':('CHECKUP1', {1},          'checkup within past year'),
    'college_grad':      ('EDUCA',    {6},          'college graduate (4+ yrs)'),
    'employed':          ('EMPLOY1',  {1, 2},       'employed for wages / self'),
    'low_income':        ('INCOME3',  {1, 2, 3, 4}, 'household income < $25,000'),
}

In [ ]:
CLEANED_PATH = "BRFSS_Cleaned/BRFSS_2024_cleaned.parquet"
df = pd.read_parquet(CLEANED_PATH).copy()
print(f"Loaded : {df.shape[0]:,} rows x {df.shape[1]} columns\n")

# STEP 1 -- how many unique values per variable
print("-- STEP 1: unique-value counts --")
for col, k in df.nunique().sort_values().items():
    print(f"  {col:<12} {k:>6} unique   | {VAR_LABELS.get(col, '')}")

# STEP 2 -- what those values are
print("\n-- STEP 2: the unique values --")
for col in df.columns:
    vals = sorted(df[col].dropna().unique().tolist())
    if len(vals) <= 12:
        print(f"  {col:<12} {vals}")
    else:
        print(f"  {col:<12} min={min(vals):g} max={max(vals):g} ({len(vals)} distinct, continuous)")

# STEP 3 -- any NaN left? drop if so
print("\n-- STEP 3: remaining NaN --")
na = df.isna().sum(); na = na[na > 0]
if na.empty:
    print("  No missing values -- nothing to drop.")
else:
    before = len(df); df = df.dropna()
    print(f"  Dropped {before - len(df):,} rows -> {len(df):,} remain.")

In [ ]:
# STEP 4 & 5 -- apply the binary recodes defined in the knobs cell above
print("-- STEP 4 & 5: create binary (0/1) variables --")
for new_col, (src, ones, meaning) in BINARY_RECODES.items():
    df[new_col] = df[src].isin(ones).astype(int)
    n1 = int(df[new_col].sum()); pct = n1 / len(df) * 100
    codes = '{' + ','.join(str(c) for c in sorted(ones)) + '}'
    print(f"  {new_col:<20} <- {src:<9} 1 if in {codes:<11} ({meaning})")
    print(f"  {'':<20}    -> 1 = {n1:,} ({pct:.1f}%) | 0 = {len(df)-n1:,} ({100-pct:.1f}%)")

# Verify one recode with a crosstab (proves the mapping is a clean split)
print("\n-- VERIFY example: _SMOKER3 -> current_smoker --")
print(pd.crosstab(df['_SMOKER3'], df['current_smoker']))

# Save the enriched dataset
OUTPUT_DIR = "BRFSS_Cleaned"
df.to_parquet(f"{OUTPUT_DIR}/BRFSS_2024_recoded.parquet", index=False)
df.to_csv(f"{OUTPUT_DIR}/BRFSS_2024_recoded.csv", index=False)
print(f"\nSaved {OUTPUT_DIR}/BRFSS_2024_recoded.parquet  ({df.shape[0]:,} x {df.shape[1]})")
df.head()

---
## Stage 3 — Exploratory data analysis  (`BRFSS_EDA.py`)

Distributions, correlations, and how each feature differs between the at-risk and no-risk
groups. **Plots render right here in the notebook** (and are also saved to `BRFSS_EDA/`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

EDA_DIR = "BRFSS_EDA"; os.makedirs(EDA_DIR, exist_ok=True)

CONTINUOUS  = ['MENTHLTH', 'PHYSHLTH', '_BMI5', '_AGE80', '_DRNKWK3']
CATEGORICAL = ['EXERANY2', '_TOTINDA', '_SMOKER3', '_BMI5CAT', 'SEXVAR',
               'EDUCA', 'INCOME3', 'EMPLOY1', '_HLTHPL2', 'PERSDOC3', 'CHECKUP1']
TARGET = 'mental_risk'

df = pd.read_parquet("BRFSS_Cleaned/BRFSS_2024_cleaned.parquet")
n = len(df)
print(f"Loaded : {n:,} rows x {df.shape[1]} columns")

# Summary statistics + target distribution
df.describe().round(3).to_csv(f"{EDA_DIR}/summary_statistics.csv")
vc = df[TARGET].value_counts()
print(f"  No Risk (0): {vc.get(0,0):,}  ({vc.get(0,0)/n*100:.1f}%)")
print(f"  At Risk (1): {vc.get(1,0):,}  ({vc.get(1,0)/n*100:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
counts = [vc.get(0, 0), vc.get(1, 0)]
bars = ax.bar(['No Risk (0)', 'At Risk (1)'], counts,
              color=['steelblue', 'tomato'], edgecolor='white', width=0.5)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, cnt+n*0.004,
            f'{cnt:,}\n({cnt/n*100:.1f}%)', ha='center', fontsize=9)
ax.set_title('Target: Mental Health Risk (>=14 poor days)'); ax.set_ylabel('Count')
plt.tight_layout(); plt.savefig(f"{EDA_DIR}/01_target_distribution.png", dpi=150); plt.show()

In [ ]:
# Continuous variable distributions
ncols = 3; nrows = (len(CONTINUOUS) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows*4)); axes = axes.flatten()
for i, col in enumerate(CONTINUOUS):
    data = df[col].dropna()
    axes[i].hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col); axes[i].set_ylabel('Count')
    axes[i].text(0.97, 0.95, f'mean={data.mean():.1f}\nmedian={data.median():.0f}',
                 transform=axes[i].transAxes, ha='right', va='top', fontsize=8,
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Continuous Variable Distributions', y=1.01)
plt.tight_layout(); plt.savefig(f"{EDA_DIR}/02_continuous_distributions.png", dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Categorical variable distributions
ncols = 3; nrows = (len(CATEGORICAL) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows*3.5)); axes = axes.flatten()
for i, col in enumerate(CATEGORICAL):
    vc2 = df[col].value_counts().sort_index()
    axes[i].bar(vc2.index.astype(str), vc2.values, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col); axes[i].set_xlabel('Code value'); axes[i].set_ylabel('Count')
for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Categorical Variable Distributions', y=1.01)
plt.tight_layout(); plt.savefig(f"{EDA_DIR}/03_categorical_distributions.png", dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Correlation matrix + correlations with the target
corr = df.corr(numeric_only=True)
corr.to_csv(f"{EDA_DIR}/correlation_matrix.csv")
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Pearson Correlation Matrix', pad=12)
plt.tight_layout(); plt.savefig(f"{EDA_DIR}/04_correlation_matrix.png", dpi=150, bbox_inches='tight'); plt.show()

print(f"Correlations with '{TARGET}' (absolute, descending):")
for feat, val in corr[TARGET].drop(TARGET).abs().sort_values(ascending=False).items():
    sign = '+' if corr.loc[feat, TARGET] > 0 else '-'
    print(f"  {feat:<15} r = {sign}{val:.4f}")

In [ ]:
# Feature distributions split by mental-health risk
features_compare = CONTINUOUS + ['_SMOKER3', 'EXERANY2', '_TOTINDA', '_BMI5CAT',
                                 'SEXVAR', 'EDUCA', 'INCOME3', 'EMPLOY1', '_HLTHPL2']
group0 = df[df[TARGET] == 0]; group1 = df[df[TARGET] == 1]
ncols = 3; nrows = (len(features_compare) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows*3.8)); axes = axes.flatten()
for i, col in enumerate(features_compare):
    ax = axes[i]
    if col in CONTINUOUS:
        ax.hist(group0[col].dropna(), bins=30, alpha=0.65, color='steelblue', label='No Risk', density=True)
        ax.hist(group1[col].dropna(), bins=30, alpha=0.65, color='tomato', label='At Risk', density=True)
        ax.set_ylabel('Density')
    else:
        vc0 = group0[col].value_counts(normalize=True).sort_index()
        vc1 = group1[col].value_counts(normalize=True).sort_index()
        idx = sorted(set(vc0.index) | set(vc1.index)); x = np.arange(len(idx)); w = 0.38
        ax.bar(x-w/2, [vc0.get(k,0) for k in idx], w, color='steelblue', label='No Risk', alpha=0.85)
        ax.bar(x+w/2, [vc1.get(k,0) for k in idx], w, color='tomato', label='At Risk', alpha=0.85)
        ax.set_xticks(x); ax.set_xticklabels([str(k) for k in idx], fontsize=8); ax.set_ylabel('Proportion')
    ax.set_title(col, fontsize=9); ax.legend(fontsize=7)
for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Feature Distributions by Mental Health Risk', y=1.01)
plt.tight_layout(); plt.savefig(f"{EDA_DIR}/05_features_by_target.png", dpi=150, bbox_inches='tight'); plt.show()

---
## Stage 4 — Train the models  (`BRFSS_Model.py`)

Trains four class-balanced classifiers — Logistic Regression, Decision Tree, Random Forest,
XGBoost. **`MENTHLTH` is excluded** from the features because it defines the target
(that would be cheating / data leakage). Trained models are saved to `BRFSS_Models/`.

**⚙️ Knobs — features & model settings.** Edit `FEATURES` to add/remove predictors,
`SCALE_COLS` for which continuous columns get standardized, and the model definitions
below for hyperparameters (tree depth, number of trees, learning rate, …).

In [ ]:
import time, pickle, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

MODELS_DIR = "BRFSS_Models"; os.makedirs(MODELS_DIR, exist_ok=True)

FEATURES = [
    'PHYSHLTH', 'EXERANY2', '_TOTINDA', '_SMOKER3', '_DRNKWK3', '_BMI5',
    '_BMI5CAT', '_AGE80', 'SEXVAR', 'EDUCA', 'INCOME3', 'EMPLOY1',
    '_HLTHPL2', 'PERSDOC3', 'CHECKUP1',
]              # NOTE: MENTHLTH intentionally excluded (it defines the target)
TARGET = 'mental_risk'

SCALE_COLS = ['PHYSHLTH', '_DRNKWK3', '_BMI5', '_AGE80']   # standardized
PASS_COLS  = [c for c in FEATURES if c not in SCALE_COLS]  # passed through as-is

In [ ]:
df = pd.read_parquet("BRFSS_Cleaned/BRFSS_2024_cleaned.parquet")
X, y = df[FEATURES], df[TARGET]
n = len(df); n1 = y.sum(); n0 = n - n1; ratio = n0 / n1
print(f"Class balance: {n0:,} no-risk ({n0/n*100:.1f}%) | {n1:,} at-risk ({n1/n*100:.1f}%)")

# 80/20 stratified split; save the indices so evaluation uses the same test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
with open(f"{MODELS_DIR}/train_test_split.pkl", 'wb') as f:
    pickle.dump({'train_index': X_train.index.tolist(), 'test_index': X_test.index.tolist()}, f)
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")

preprocessor = ColumnTransformer(transformers=[
    ('scale', StandardScaler(), SCALE_COLS),
    ('pass',  'passthrough',    PASS_COLS),
], remainder='drop')

In [ ]:
# Model definitions -- class_weight='balanced' handles the 86/14 imbalance;
# XGBoost uses scale_pos_weight instead. Edit any hyperparameter here.
models = {
    'Logistic_Regression': Pipeline([('prep', preprocessor),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000,
                                     random_state=42, solver='lbfgs', n_jobs=-1))]),
    'Decision_Tree': Pipeline([('prep', preprocessor),
        ('model', DecisionTreeClassifier(class_weight='balanced', max_depth=8,
                                         min_samples_leaf=50, random_state=42))]),
    'Random_Forest': Pipeline([('prep', preprocessor),
        ('model', RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                         max_depth=12, min_samples_leaf=20,
                                         max_features='sqrt', random_state=42, n_jobs=-1))]),
    'XGBoost': Pipeline([('prep', preprocessor),
        ('model', XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8, scale_pos_weight=ratio,
                                use_label_encoder=False, eval_metric='logloss',
                                random_state=42, n_jobs=-1))]),
}

In [ ]:
# Train + evaluate each model (Random Forest takes the longest, ~a minute)
results = []
for name, pipe in models.items():
    t0 = time.time(); pipe.fit(X_train, y_train); elapsed = time.time() - t0
    tr_acc = accuracy_score(y_train, pipe.predict(X_train))
    tr_auc = roc_auc_score(y_train, pipe.predict_proba(X_train)[:, 1])
    te_pred = pipe.predict(X_test); te_prob = pipe.predict_proba(X_test)[:, 1]
    te_acc = accuracy_score(y_test, te_pred); te_auc = roc_auc_score(y_test, te_prob)
    cm = confusion_matrix(y_test, te_pred)
    print(f"[{name}]  fit {elapsed:.1f}s  |  Train AUC {tr_auc:.4f}  |  Test AUC {te_auc:.4f}  Acc {te_acc:.4f}")
    results.append({'model': name, 'fit_time_s': round(elapsed, 2),
                    'train_acc': round(tr_acc, 4), 'train_auc': round(tr_auc, 4),
                    'test_acc': round(te_acc, 4), 'test_auc': round(te_auc, 4),
                    'test_tn': int(cm[0,0]), 'test_fp': int(cm[0,1]),
                    'test_fn': int(cm[1,0]), 'test_tp': int(cm[1,1])})
    with open(f"{MODELS_DIR}/{name}.pkl", 'wb') as f:
        pickle.dump(pipe, f)

results_df = pd.DataFrame(results).set_index('model')
results_df.to_csv(f"{MODELS_DIR}/model_comparison.csv")
print(f"\nBest Test AUC: {results_df['test_auc'].idxmax()} ({results_df['test_auc'].max():.4f})")
results_df[['train_acc','train_auc','test_acc','test_auc','fit_time_s']]

---
## Stage 5 — Evaluate  (`BRFSS_Evaluate.py`)

Reloads the saved models and the held-out test set, then builds the metrics table and the
ROC / precision-recall / confusion-matrix / comparison charts (shown inline and saved to
`BRFSS_Evaluation/`).

In [ ]:
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay)

EVAL_DIR = "BRFSS_Evaluation"; os.makedirs(EVAL_DIR, exist_ok=True)
MODEL_NAMES = ['Logistic_Regression', 'Decision_Tree', 'Random_Forest', 'XGBoost']
COLORS = ['royalblue', 'darkorange', 'forestgreen', 'crimson']

# Rebuild the exact same test set from the saved split indices
df = pd.read_parquet("BRFSS_Cleaned/BRFSS_2024_cleaned.parquet")
with open(f"{MODELS_DIR}/train_test_split.pkl", 'rb') as f:
    test_idx = pickle.load(f)['test_index']
X_test = df.loc[test_idx, FEATURES]; y_test = df.loc[test_idx, TARGET]

pipelines, y_preds, y_probs = {}, {}, {}
for name in MODEL_NAMES:
    with open(f"{MODELS_DIR}/{name}.pkl", 'rb') as f:
        pipelines[name] = pickle.load(f)
    y_preds[name] = pipelines[name].predict(X_test)
    y_probs[name] = pipelines[name].predict_proba(X_test)[:, 1]
print(f"Test set: {len(X_test):,} rows ({y_test.sum():,} at-risk)")

In [ ]:
# Metrics table
rows = []
for name in MODEL_NAMES:
    yp, ypr = y_preds[name], y_probs[name]
    tn, fp, fn, tp = confusion_matrix(y_test, yp).ravel()
    rows.append({'Model': name.replace('_', ' '),
        'Accuracy': round(accuracy_score(y_test, yp), 4),
        'ROC-AUC': round(roc_auc_score(y_test, ypr), 4),
        'Avg-PR-AUC': round(average_precision_score(y_test, ypr), 4),
        'F1 (At Risk)': round(f1_score(y_test, yp), 4),
        'Recall (AR)': round(tp/(tp+fn), 4),
        'Precision(AR)': round(tp/(tp+fp), 4),
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp})
metrics_df = pd.DataFrame(rows).set_index('Model')
metrics_df.to_csv(f"{EVAL_DIR}/metrics_table.csv")
print("Best ROC-AUC:", metrics_df['ROC-AUC'].idxmax(), f"({metrics_df['ROC-AUC'].max()})")
metrics_df[['Accuracy','ROC-AUC','Avg-PR-AUC','F1 (At Risk)','Recall (AR)','Precision(AR)']]

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(7, 6))
for name, color in zip(MODEL_NAMES, COLORS):
    fpr, tpr, _ = roc_curve(y_test, y_probs[name])
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f"{name.replace('_',' ')} (AUC={roc_auc_score(y_test, y_probs[name]):.3f})")
ax.plot([0,1], [0,1], 'k--', lw=1, label='Random (0.500)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models'); ax.legend(loc='lower right', fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{EVAL_DIR}/01_roc_curves.png", dpi=150); plt.show()

In [ ]:
# Precision-Recall curves
fig, ax = plt.subplots(figsize=(7, 6))
ax.axhline(y_test.mean(), color='k', linestyle='--', lw=1, label=f'Baseline ({y_test.mean():.3f})')
for name, color in zip(MODEL_NAMES, COLORS):
    prec, rec, _ = precision_recall_curve(y_test, y_probs[name])
    ax.plot(rec, prec, color=color, lw=2,
            label=f"{name.replace('_',' ')} (AP={average_precision_score(y_test, y_probs[name]):.3f})")
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — All Models'); ax.legend(loc='upper right', fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{EVAL_DIR}/02_precision_recall_curves.png", dpi=150); plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10)); axes = axes.flatten()
for i, name in enumerate(MODEL_NAMES):
    cm = confusion_matrix(y_test, y_preds[name])
    acc = accuracy_score(y_test, y_preds[name]); auc = roc_auc_score(y_test, y_probs[name])
    ConfusionMatrixDisplay(cm, display_labels=['No Risk', 'At Risk']).plot(
        ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(f"{name.replace('_',' ')}\nAcc={acc:.3f}  AUC={auc:.3f}", fontsize=10)
plt.suptitle('Confusion Matrices (Test Set)', y=1.01)
plt.tight_layout(); plt.savefig(f"{EVAL_DIR}/03_confusion_matrices.png", dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Metric comparison bar chart
plot_metrics = ['Accuracy', 'ROC-AUC', 'F1 (At Risk)', 'Recall (AR)']
x = np.arange(len(plot_metrics)); width = 0.18
fig, ax = plt.subplots(figsize=(11, 5))
for i, (name, color) in enumerate(zip(MODEL_NAMES, COLORS)):
    vals = [metrics_df.loc[name.replace('_', ' '), m] for m in plot_metrics]
    bars = ax.bar(x + i*width, vals, width, label=name.replace('_', ' '), color=color, alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003, f'{v:.3f}',
                ha='center', va='bottom', fontsize=7, rotation=90)
ax.set_xticks(x + width*(len(MODEL_NAMES)-1)/2); ax.set_xticklabels(plot_metrics)
ax.set_ylabel('Score'); ax.set_ylim(0, 1.05); ax.set_title('Model Performance Comparison (Test Set)')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f"{EVAL_DIR}/04_metric_comparison.png", dpi=150); plt.show()

---
## (Optional) Keep the outputs

Colab deletes files when the runtime ends. To save what this notebook regenerated, either
download a file or copy the output folders to your Google Drive.

In [ ]:
# --- Download a single file to your computer ---
# from google.colab import files
# files.download('BRFSS_Cleaned/BRFSS_2024_recoded.csv')

# --- Or copy all outputs to Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/BRFSS_outputs
# !cp -r BRFSS_Cleaned BRFSS_Models BRFSS_Evaluation BRFSS_EDA /content/drive/MyDrive/BRFSS_outputs/
# print('Copied outputs to Drive.')